# 01 — Schema and Ingest

**Purpose:** Create the Neo4j schema (constraints + indexes), define seed story data, and populate the graph using the `GraphClient` MERGE helpers.

**Prerequisites:**
- Neo4j Desktop running with APOC enabled (bolt://localhost:7687)
- `.env` file configured with `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `pip install -r requirements.txt` completed

**Expected outputs:**
- Neo4j database with uniqueness constraints and composite indexes
- Seed graph: 3 characters, 2 locations, 3 events, relationships
- Pyvis visualization of the seed graph

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

from src.graph_client import GraphClient
from src.schema import (
    Character, Location, Event, StoryObject,
    Segment, Relationship, SessionState,
)

In [ ]:
gc = GraphClient(
    uri=os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    user=os.getenv("NEO4J_USER", "neo4j"),
    password=os.getenv("NEO4J_PASSWORD", ""),
)
gc.verify_connectivity()
print("Connected to Neo4j")

## 1. Clear existing data and create constraints

In [ ]:
gc.clear_database()
gc.create_constraints()
print("Database cleared and constraints created")

## 2. Define seed story data

A small fantasy scenario with 3 characters, 2 locations, 1 object, and 3 events.
This provides enough structure to validate all MERGE paths and relationship types.

In [ ]:
BRANCH = "main"

# --- Locations ---
locations = [
    Location(name="Iron Tavern", type="tavern", accessible=True,
             description_summary="A smoky tavern at the crossroads, known for strong ale and loose talk."),
    Location(name="Thornwood Bridge", type="bridge", accessible=True,
             description_summary="A crumbling stone bridge over the Ashfall River. Ambush territory."),
]

# --- Characters ---
characters = [
    Character(name="Kael", status="alive", current_location_id="Iron Tavern",
              alignment="neutral", traits=["cautious", "resourceful"]),
    Character(name="Elara", status="alive", current_location_id="Iron Tavern",
              alignment="good", traits=["brave", "impulsive", "silver-tongued"]),
    Character(name="Maren", status="alive", current_location_id="Thornwood Bridge",
              alignment="hostile", traits=["cunning", "ruthless"]),
]

# --- Object ---
objects = [
    StoryObject(name="Shadow Dagger", current_owner_id="Kael",
                significance="A cursed blade that whispers to its wielder.",
                last_seen_location_id="Iron Tavern"),
]

# --- Events ---
events = [
    Event(description="Kael and Elara meet at the Iron Tavern to plan a journey north.",
          seq_id=1, branch_id=BRANCH, outcome="alliance_formed"),
    Event(description="Maren ambushes travelers at Thornwood Bridge, stealing supplies.",
          seq_id=2, branch_id=BRANCH, outcome="travelers_robbed"),
    Event(description="Kael learns of Maren's ambush from a wounded traveler at the tavern.",
          seq_id=3, branch_id=BRANCH, outcome="intel_gained"),
]

# --- Segments (the narrative text for each event) ---
segments = [
    Segment(text="The fire crackled in the hearth as Kael slid into the booth across from Elara. "
            "'We head north at dawn,' he said, keeping his voice low. She nodded, fingers drumming "
            "the table. 'And if the bridge is watched?' 'Then we find another way.'",
            seq_id=1, branch_id=BRANCH),
    Segment(text="Maren stepped from the treeline, blade drawn. The merchant's cart lurched to a stop. "
            "'Toll,' she said, without preamble. The driver's hands were shaking before she finished the word.",
            seq_id=2, branch_id=BRANCH),
    Segment(text="The wounded man stumbled through the tavern door, clutching his side. 'The bridge,' "
            "he gasped. 'A woman — dark hair, moves like a ghost.' Kael's jaw tightened. He knew that "
            "description.",
            seq_id=3, branch_id=BRANCH),
]

print(f"Seed data: {len(characters)} characters, {len(locations)} locations, "
      f"{len(objects)} objects, {len(events)} events, {len(segments)} segments")

## 3. Ingest seed data into Neo4j

In [ ]:
for loc in locations:
    created = gc.merge_location(loc)
    print(f"  Location '{loc.name}': {'created' if created else 'updated'}")

for char in characters:
    created = gc.merge_character(char, BRANCH)
    print(f"  Character '{char.name}': {'created' if created else 'updated'}")

for obj in objects:
    created = gc.merge_object(obj, BRANCH)
    print(f"  Object '{obj.name}': {'created' if created else 'updated'}")

for evt in events:
    created = gc.merge_event(evt)
    print(f"  Event seq_id={evt.seq_id}: {'created' if created else 'updated'}")

for seg in segments:
    created = gc.merge_segment(seg)
    print(f"  Segment seq_id={seg.seq_id}: {'created' if created else 'updated'}")

print("\nAll seed nodes ingested.")

## 4. Create relationships

In [ ]:
relationships = [
    # LOCATED_AT
    Relationship(type="LOCATED_AT", source="Kael", target="Iron Tavern",
                 properties={"as_of_seq_id": 1}),
    Relationship(type="LOCATED_AT", source="Elara", target="Iron Tavern",
                 properties={"as_of_seq_id": 1}),
    Relationship(type="LOCATED_AT", source="Maren", target="Thornwood Bridge",
                 properties={"as_of_seq_id": 2}),
    # KNOWS
    Relationship(type="KNOWS", source="Kael", target="Elara",
                 properties={"sentiment": "allied", "since_event_id": 1}),
    Relationship(type="KNOWS", source="Kael", target="Maren",
                 properties={"sentiment": "hostile", "since_event_id": 3}),
    # PARTICIPATED_IN (Character -> Event via seq_id)
    Relationship(type="PARTICIPATED_IN", source="Kael", target="1",
                 properties={"role": "protagonist"}),
    Relationship(type="PARTICIPATED_IN", source="Elara", target="1",
                 properties={"role": "protagonist"}),
    Relationship(type="PARTICIPATED_IN", source="Maren", target="2",
                 properties={"role": "antagonist"}),
    Relationship(type="PARTICIPATED_IN", source="Kael", target="3",
                 properties={"role": "witness"}),
    # CAUSED_BY (Event -> Event via seq_id)
    Relationship(type="CAUSED_BY", source="3", target="2",
                 properties={}),
    # OWNS
    Relationship(type="OWNS", source="Kael", target="Shadow Dagger",
                 properties={"since_event_id": 1}),
]

for rel in relationships:
    ok = gc.merge_relationship(rel, BRANCH)
    print(f"  {rel.type}: {rel.source} -> {rel.target}: {'OK' if ok else 'FAILED'}")

print(f"\n{len(relationships)} relationships created.")

## 4b. Structural enrichment

Create REFERENCES_GRAPH_STATE links from seed segments to their mentioned entities, link segments to their events, and run `enrich_structural_edges` to create any LOCATED_AT/OWNS edges implied by node properties.

In [ ]:
seed_refs = {
    1: ["Kael", "Elara", "Iron Tavern"],
    2: ["Maren", "Thornwood Bridge"],
    3: ["Kael", "Maren", "Iron Tavern", "Thornwood Bridge"],
}
for seq_id, names in seed_refs.items():
    gc.create_references_graph_state(seq_id, BRANCH, names)
    print(f"  Segment #{seq_id} -> {names}")

enrich_counts = gc.enrich_structural_edges(BRANCH)
print(f"\nStructural enrichment: {enrich_counts}")
print("Seed segments and structural edges linked.")

## 5. Validate: node and relationship counts

In [ ]:
node_counts = gc.get_node_counts()
rel_counts = gc.get_relationship_counts()

print("Node counts:")
for label, count in sorted(node_counts.items()):
    print(f"  {label}: {count}")

print("\nRelationship counts:")
for rtype, count in sorted(rel_counts.items()):
    print(f"  {rtype}: {count}")

print("\nGraph summary facts:")
for fact in gc.get_graph_summary_facts(BRANCH):
    print(f"  - {fact}")

## 6. Visualize the seed graph with pyvis

In [ ]:
from pyvis.network import Network
from IPython.display import HTML
import tempfile, os

COLOR_MAP = {
    "Character": "#4A90D9",
    "Location": "#27AE60",
    "Event": "#E67E22",
    "Object": "#95A5A6",
    "Segment": "#8E44AD",
    "Faction": "#C0392B",
}

net = Network(height="500px", width="100%", notebook=True, directed=True)
net.barnes_hut()

vis_query = """
MATCH (n)
OPTIONAL MATCH (n)-[r]->(m)
RETURN n, labels(n) AS n_labels, r, type(r) AS r_type, m, labels(m) AS m_labels
"""

seen_nodes = set()
with gc._driver.session(database=gc._database) as session:
    for record in session.run(vis_query):
        n = record["n"]
        n_labels = record["n_labels"]
        n_id = n.element_id
        if n_id not in seen_nodes:
            label = n_labels[0] if n_labels else "Unknown"
            name = dict(n).get("name", dict(n).get("description", f"seq:{dict(n).get('seq_id', '?')}"))
            if len(name) > 40:
                name = name[:37] + "..."
            net.add_node(n_id, label=name, color=COLOR_MAP.get(label, "#BDC3C7"),
                        title=f"{label}: {name}")
            seen_nodes.add(n_id)

        m = record["m"]
        if m is not None:
            m_id = m.element_id
            m_labels = record["m_labels"]
            if m_id not in seen_nodes:
                label = m_labels[0] if m_labels else "Unknown"
                name = dict(m).get("name", dict(m).get("description", f"seq:{dict(m).get('seq_id', '?')}"))
                if len(name) > 40:
                    name = name[:37] + "..."
                net.add_node(m_id, label=name, color=COLOR_MAP.get(label, "#BDC3C7"),
                            title=f"{label}: {name}")
                seen_nodes.add(m_id)

            r_type = record["r_type"]
            net.add_edge(n_id, m_id, label=r_type, title=r_type)

html_path = os.path.join(tempfile.gettempdir(), "lorekeeper_seed_graph.html")
net.show(html_path)
print(f"Graph visualization saved to {html_path}")
HTML(filename=html_path)

In [ ]:
gc.close()
print("Done. GraphClient connection closed.")